# Molecular Dynamics of a Protein–Ligand Complex with OpenMM

Molecular dynamics (MD) follows the motion of every atom in a system by integrating Newton's equations of motion, with forces from a *force field*. This notebook takes your refined crystal structure and a 3D model of its ligand, surrounds them with water and ions, and runs a short MD simulation with [OpenMM](https://openmm.org).

Force fields: **AMBER ff14SB** for the protein, **TIP3P** for water, and **OpenFF Sage** for the ligand, assigned automatically by the `openmmforcefields` package.

By the end of this tutorial, you will be able to:

1. Prepare a protein from a PDB file: rebuild missing loops and atoms, and add hydrogens.
2. Load a ligand from an SDF file and assign it a force field.
3. Combine protein and ligand and solvate them in a box of water and ions.
4. Energy-minimize the system and run a short MD simulation.
5. Plot the energies and temperature, visualize the trajectory, and track the distance between two atoms.
   
Settings used here: 300 K, 2 fs time step, 50,000 steps = 100 ps.

#### **Some of these steps (energy minimization and MD simulation) take 10s seconds to minutes to run**

## Setup

Run this notebook on Google Colab with a **T4 GPU** runtime (*Runtime → Change runtime type*).

The packages come from conda-forge, so the first cell installs conda into Colab. **When it finishes, the runtime restarts** and Colab reports that the session crashed; that is expected. Then run the second cell, which installs the packages (a few minutes).

In [ ]:
!pip install -q condacolab
import condacolab
condacolab.install()

In [ ]:
!mamba install -q -y -c conda-forge openmm pdbfixer openmmforcefields mdtraj nglview
from google.colab import output
output.enable_custom_widget_manager()

In [ ]:
import openmm
import openmm.app as app
import openmm.unit as unit
from pdbfixer import PDBFixer
from openff.toolkit import Molecule, Topology
from openmmforcefields.generators import SMIRNOFFTemplateGenerator
import mdtraj as md
import nglview
import pandas as pd

## 1. Input files

Upload two files to Colab (folder icon on the left):

- **Your refined structure** as a PDB file. Waters, ions and the ligand in this file are removed below; only the protein is used. Also remove any redundant chains.  
- **Your ligand** as an SDF file with hydrogens, in its docked position (same coordinate frame as the protein). You can make it in Avogadro from the ligand coordinates in your PDB file.
- `pdb_id` is the PDB entry your structure came from. Its full sequence is used to find missing loops.


In [ ]:
# your PDB file (upload it to Colab)
protein_path = "protein.pdb"
# your ligand file (upload it to Colab)
ligand_path = "ligand.sdf"
# PDB_ID to get protein sequence from RCSB PDB
pdb_id = "1hvy" 

## 2. The protein

[PDBFixer](https://github.com/openmm/pdbfixer) cleans up the crystal structure: it removes waters, finds residues that are missing compared with the full sequence and rebuilds them, adds missing heavy atoms (e.g. truncated side chains), and adds hydrogens.


In [ ]:
fixer = PDBFixer(filename=protein_path)
fixer.removeHeterogens(keepWater=False)
fixer.sequences = PDBFixer(pdbid=pdb_id).sequences
fixer.findMissingResidues()
print("Missing residues to rebuild:", fixer.missingResidues)
fixer.findMissingAtoms()
fixer.addMissingAtoms()
fixer.addMissingHydrogens(7.0)

## 3. The ligand

The ligand is read from the SDF file and partial charges are generated (via a empirical method). The ligand force field parameters are assigned using OpenFF Sage force field.

In [ ]:
ligands = Topology.from_molecules(Molecule.from_file(ligand_path, allow_undefined_stereo=True))
for ligand in ligands.molecules:
    ligand.assign_partial_charges('gasteiger')

## 4. Build and solvate the complex

The force field combines AMBER ff14SB, TIP3P water, and a generator that parametrizes the ligand with OpenFF Sage. The ligand is added to the protein model, then a box of water with at least 1 nm between the protein and the box edge and 0.15 M NaCl. The result is saved as `complex_solvated.pdb`. A warning about virtual sites can be ignored.

In [ ]:
forcefield = app.ForceField("amber14-all.xml", "amber14/tip3p.xml")
forcefield.registerTemplateGenerator(SMIRNOFFTemplateGenerator(molecules=list(ligands.molecules), forcefield="openff-2.3.0").generator)
model = app.Modeller(fixer.topology, fixer.positions)
model.add(ligands.to_openmm(), ligands.get_positions().to_openmm())
model.addSolvent(forcefield, padding=1.0 * unit.nanometer, ionicStrength=0.15 * unit.molar)
app.PDBFile.writeFile(model.topology, model.positions, open("complex_solvated.pdb", "w"), keepIds=True)

## 5. Set up the simulation

`createSystem` assigns force-field parameters to every atom. Long-range electrostatics use particle-mesh Ewald (PME), and bonds to hydrogen are constrained, which allows a 2 fs time step. A Langevin integrator keeps the temperature at 300 K.

In [ ]:
system = forcefield.createSystem(model.topology, nonbondedMethod=app.PME, constraints=app.HBonds)
integrator = openmm.LangevinMiddleIntegrator(300 * unit.kelvin, 1 / unit.picosecond, 0.002 * unit.picoseconds)
simulation = app.Simulation(model.topology, system, integrator)
simulation.context.setPositions(model.positions)

## 6. Energy minimization

Minimization relieves clashes and strained geometry from the added atoms, water and ions before dynamics starts. The potential energy should drop a lot.

In [ ]:
print("Before:", simulation.context.getState(getEnergy=True).getPotentialEnergy())
simulation.minimizeEnergy()
print("After: ", simulation.context.getState(getEnergy=True).getPotentialEnergy())

## 7. Run the simulation

Reporters save coordinates to `trajectory.dcd` and energies and temperature to `log.csv` every 100 steps. Then 50,000 steps of 2 fs = 100 ps are run. This takes several minutes on a T4 GPU and much longer on a CPU.

Modern MD simulations are ran on the 100s ns - 10s microsecond time scale

In [ ]:
simulation.reporters.append(app.DCDReporter("trajectory.dcd", 500))
simulation.reporters.append(app.StateDataReporter("log.csv", 500, time=True, potentialEnergy=True, kineticEnergy=True,
                                                  totalEnergy=True, temperature=True))
simulation.step(50000)

## 8. Energies and temperature

Each column of `log.csv` is plotted against time. The temperature should fluctuate around 300 K.

In [ ]:
log = pd.read_csv("log.csv", index_col=0).rename_axis("Time (ps)")
log.plot(subplots=True, layout=(2, 2), figsize=(10, 6))

## 9. Visualize the trajectory

MDTraj loads the trajectory and `image_molecules` puts molecules that drifted across the periodic box back together. Water is hidden. Hover over an atom to see its name and residue; the ligand residue is called `UNK`.

In [ ]:
traj = md.load("trajectory.dcd", top=md.Topology.from_openmm(simulation.topology)).image_molecules()
nglview.show_mdtraj(traj.atom_slice(traj.topology.select("protein or resname UNK")))

## 10. Track a distance

Pick two atoms with the MDTraj [selection syntax](https://mdtraj.org/1.9.4/atom_selection.html), for example a ligand atom and a residue it interacts with (`residue` uses the numbering of your PDB file). Frames were saved every 500 steps of 2 fs, so one frame is 1 ps.

In [ ]:
atom_1 = traj.topology.select("resname UNK and not element H")[0]  # first heavy atom of the ligand; e.g. "resname UNK and name N1x"
atom_2 = traj.topology.select("protein and name CA")[0]             # first Cα of the protein; e.g. "residue 25 and name CG"
distance = md.compute_distances(traj, [[atom_1, atom_2]])[:, 0] * 10  # nm -> Å
pd.Series(distance, index=traj.time * 0.2).plot(xlabel="Time (ps)", ylabel=f"{traj.topology.atom(atom_1)} – {traj.topology.atom(atom_2)} (Å)");

## 11. Save the final structure

Download `complex_final.pdb`, `trajectory.dcd` and `log.csv` for your report.

In [ ]:
positions = simulation.context.getState(getPositions=True).getPositions()
app.PDBFile.writeFile(simulation.topology, positions, open("complex_final.pdb", "w"), keepIds=True)